# Hypotes – geografins påverkan på bostadspriset

Den här notebooken undersöker om geografiska variabler förbättrar modellens förmåga att uppskatta utgångspriset.

Vi jämför två versioner av samma modell:

1. En modell utan kommun, latitud och longitud.
2. Den befintliga globalmodellen med geografiska variabler.

Modellerna använder samma tränings- och testbostäder. Den huvudsakliga skillnaden är därför om geografiska features ingår.

Notebooken skapar även två filer som senare används av Streamlit:

- `data/geography_metrics.json`
- `data/municipality_profiles.csv`


## Importera paket

Vi använder samma scikit-learn-komponenter som i modellträningen. Den färdigtränade globalmodellen laddas med Joblib.


In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import median_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## Sökvägar

Notebooken förutsätter att den ligger i projektets `notebooks`-mapp.


In [ ]:
DATA_DIR = Path("../data")
MODELS_DIR = Path("../models")

CLEANED_DATA_PATH = DATA_DIR / "cleaned_housing_data.parquet"
GLOBAL_MODEL_PATH = MODELS_DIR / "global_model.joblib"
METRICS_PATH = DATA_DIR / "geography_metrics.json"
PROFILES_PATH = DATA_DIR / "municipality_profiles.csv"


## Läs den tvättade datan

Datan har redan tvättats i `model_training.ipynb`. På så sätt slipper vi duplicera reglerna för datatvätt här.


In [ ]:
if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(
        "Kör Parquet-exporten i model_training.ipynb först. "
        f"Filen saknas: {CLEANED_DATA_PATH}"
    )

if not GLOBAL_MODEL_PATH.exists():
    raise FileNotFoundError(
        "Den sparade globalmodellen saknas. "
        f"Filen saknas: {GLOBAL_MODEL_PATH}"
    )

df = pd.read_parquet(CLEANED_DATA_PATH)

print(f"Antal tvättade bostäder: {len(df)}")
df.head()


## Återskapa samma datauppdelning

Samma `random_state`, teststorlek och stratifiering används som i modellträningen. Därför får hypotesen samma train-, validation- och testbostäder.


In [ ]:
train_validation_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["typology"]
)

train_df, validation_df = train_test_split(
    train_validation_df,
    test_size=0.25,
    random_state=42,
    stratify=train_validation_df["typology"]
)

final_train_df = pd.concat(
    [train_df, validation_df],
    ignore_index=True
)

print(f"Slutlig träningsdata: {len(final_train_df)}")
print(f"Testdata: {len(test_df)}")


## Funktion för utvärdering

Vi använder MAE, RMSE, medianfel och R², precis som i modellträningen.


In [ ]:
def calculate_metrics(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "Median error": median_absolute_error(y_true, predictions),
        "R2": r2_score(y_true, predictions)
    }


## Ladda modellen med geografi

Globalmodellen innehåller redan kommun, latitud och longitud. Vi återanvänder både den färdigtränade pipelinen och dess metadata.


In [ ]:
global_bundle = joblib.load(GLOBAL_MODEL_PATH)

global_model = global_bundle["pipeline"]
global_metadata = global_bundle["metadata"]

with_geography_features = global_metadata["feature_columns"]

print("Modell:", global_metadata["model_name"])
print("Features med geografi:", with_geography_features)


## Välj features utan geografi

Modellen utan geografi får fortfarande information om bostadens storlek, antal rum, tomt och bostadstyp. Kommun och koordinater tas bort.


In [ ]:
without_geography_numeric_features = [
    "land_area_sqm",
    "living_area_sqm",
    "number_rooms",
    "has_land_area"
]

without_geography_categorical_features = [
    "typology"
]

without_geography_features = (
    without_geography_numeric_features
    + without_geography_categorical_features
)

without_geography_features


## Preprocessing utan geografi

Numeriska saknade värden fylls med medianen. Bostadstypen one-hot-kodas.


In [ ]:
without_geography_preprocessor = ColumnTransformer([
    (
        "numeric",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        ),
        without_geography_numeric_features
    ),
    (
        "categorical",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ),
        without_geography_categorical_features
    )
])


## Träna modellen utan geografi

Vi klonar estimatorn från den färdiga globalmodellen. Därmed återanvänds samma modelltyp och hyperparametrar. Bara preprocessing och feature-listan skiljer sig.


In [ ]:
without_geography_model = Pipeline([
    (
        "preprocessor",
        without_geography_preprocessor
    ),
    (
        "model",
        clone(global_model.named_steps["model"])
    )
])

without_geography_model.fit(
    final_train_df[without_geography_features],
    final_train_df["asking_price_sek"]
)


## Gör prediktioner på samma testdata

Ingen av modellerna tränas på testdatan. Båda utvärderas på exakt samma bostäder.


In [ ]:
without_geography_predictions = without_geography_model.predict(
    test_df[without_geography_features]
)

with_geography_predictions = global_model.predict(
    test_df[with_geography_features]
)

without_geography_metrics = calculate_metrics(
    test_df["asking_price_sek"],
    without_geography_predictions
)

with_geography_metrics = calculate_metrics(
    test_df["asking_price_sek"],
    with_geography_predictions
)


## Jämför resultaten

Lägre MAE och RMSE är bättre. Högre R² är bättre.


In [ ]:
geography_comparison = pd.DataFrame([
    {
        "Model": "Utan geografi",
        **without_geography_metrics
    },
    {
        "Model": "Med geografi",
        **with_geography_metrics
    }
])

geography_comparison


## Beräkna förbättringen

En positiv procentsats innebär att modellen med geografi har lägre RMSE.


In [ ]:
rmse_without_geography = without_geography_metrics["RMSE"]
rmse_with_geography = with_geography_metrics["RMSE"]

rmse_improvement_percent = (
    (rmse_without_geography - rmse_with_geography)
    / rmse_without_geography
    * 100
)

print(
    "RMSE utan geografi:",
    f"{rmse_without_geography:,.0f} kr"
)

print(
    "RMSE med geografi:",
    f"{rmse_with_geography:,.0f} kr"
)

print(
    "Förbättring:",
    f"{rmse_improvement_percent:.1f} %"
)


## Resultat per bostadstyp

Den här tabellen visar om geografin hjälper olika mycket för lägenheter, villor och radhus.


In [ ]:
segment_results = []

for typology in ["APARTMENT", "HOUSE", "ROW_HOUSE"]:
    segment_mask = test_df["typology"] == typology
    segment_target = test_df.loc[
        segment_mask,
        "asking_price_sek"
    ]

    metrics_without = calculate_metrics(
        segment_target,
        without_geography_predictions[segment_mask]
    )

    metrics_with = calculate_metrics(
        segment_target,
        with_geography_predictions[segment_mask]
    )

    segment_improvement = (
        (metrics_without["RMSE"] - metrics_with["RMSE"])
        / metrics_without["RMSE"]
        * 100
    )

    segment_results.append({
        "Bostadstyp": typology,
        "RMSE utan geografi": metrics_without["RMSE"],
        "RMSE med geografi": metrics_with["RMSE"],
        "Förbättring (%)": segment_improvement
    })

segment_comparison = pd.DataFrame(segment_results)
segment_comparison


## Skapa kommunprofiler

Varje kombination av kommun och bostadstyp får representativa koordinater, antal observationer och medianpris per kvadratmeter.

Kommuner med få observationer sparas fortfarande i filen. Streamlit-sidan kan senare kräva minst 20 observationer innan en kommun visas.


In [ ]:
profile_data = df.copy()

profile_data["calculated_sqm_price"] = (
    profile_data["asking_price_sek"]
    / profile_data["living_area_sqm"]
)

municipality_profiles = (
    profile_data
    .groupby(
        ["municipality", "typology"],
        as_index=False
    )
    .agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        observations=("asking_price_sek", "size"),
        median_asking_price=("asking_price_sek", "median"),
        median_sqm_price=("calculated_sqm_price", "median")
    )
    .sort_values(
        ["typology", "municipality"]
    )
)

municipality_profiles.head()


## Spara resultat till appen

Streamlit behöver bara läsa de färdiga filerna. Appen behöver därför inte träna modeller eller bearbeta hela datasetet vid uppstart.


In [ ]:
def json_metrics(metrics):
    return {
        "mae": float(metrics["MAE"]),
        "rmse": float(metrics["RMSE"]),
        "median_error": float(metrics["Median error"]),
        "r2": float(metrics["R2"])
    }

metrics_to_save = {
    "without_geography": json_metrics(
        without_geography_metrics
    ),
    "with_geography": json_metrics(
        with_geography_metrics
    ),
    "rmse_improvement_percent": float(
        rmse_improvement_percent
    ),
    "test_rows": int(len(test_df)),
    "random_state": 42,
    "segments": {
        row["Bostadstyp"]: {
            "rmse_without_geography": float(
                row["RMSE utan geografi"]
            ),
            "rmse_with_geography": float(
                row["RMSE med geografi"]
            ),
            "improvement_percent": float(
                row["Förbättring (%)"]
            )
        }
        for row in segment_results
    }
}

with METRICS_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        metrics_to_save,
        file,
        ensure_ascii=False,
        indent=2
    )

municipality_profiles.to_csv(
    PROFILES_PATH,
    index=False,
    encoding="utf-8"
)

print(f"Sparad: {METRICS_PATH}")
print(f"Sparad: {PROFILES_PATH}")


## Slutsats

Om modellen med geografi får tydligt lägre RMSE ger resultatet stöd för hypotesen att kommun och koordinater bidrar till modellens prediktionsförmåga.

Resultatet visar ett prediktivt samband. Det bevisar inte att geografin ensam orsakar hela prisskillnaden. Datasetet saknar exempelvis information om skick, våningsplan, månadsavgift, skolor och kollektivtrafik.
